# Notebook 02 - Classification Training and Evaluation

**Project:** Explainable Deep Learning for MRI Brain Tumor Classification and Segmentation Using Transfer Learning, MONAI, U-Net, and Grad-CAM

Run notebooks in order. Each notebook writes outputs into the same project folder so later notebooks can reuse them.


## Purpose

This notebook trains and evaluates classification models:

- Basic CNN baseline
- ResNet50 transfer learning
- Optional DenseNet121 / EfficientNet-B0

It saves model checkpoints, training curves, confusion matrices, classification reports, and a model comparison table.


In [ ]:
import sys, subprocess, os, json, random, copy
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "torch", "torchvision", "scikit-learn", "pandas",
                           "numpy", "matplotlib", "pillow", "tqdm"])
print("Running in Colab:", IN_COLAB)


In [ ]:
from pathlib import Path
import json, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_auc_score
)
from sklearn.preprocessing import label_binarize

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
if IN_COLAB:
    PROJECT_ROOT = Path("/content/brain_tumor_xai_project")
else:
    PROJECT_ROOT = Path.cwd() / "brain_tumor_xai_project"

config = json.loads((PROJECT_ROOT / "project_config.json").read_text())
PROJECT_ROOT = Path(config["project_root"])
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = Path(config["model_dir"]); MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = Path(config["figure_dir"]); FIGURE_DIR.mkdir(parents=True, exist_ok=True)

metadata = pd.read_csv(config["classification_metadata_csv"])
class_names = json.loads(Path(config["class_names_json"]).read_text())
num_classes = len(class_names)
IMAGE_SIZE = int(config.get("image_size", 224))

print("Classes:", class_names)
print(metadata.groupby(["split", "class_name"]).size().unstack(fill_value=0))


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2 if IN_COLAB else 0

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class MRIDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        return self.transform(img), int(row["label"]), row["path"]

train_ds = MRIDataset(metadata[metadata["split"]=="train"], train_transform)
val_ds = MRIDataset(metadata[metadata["split"]=="val"], eval_transform)
test_ds = MRIDataset(metadata[metadata["split"]=="test"], eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Train/Val/Test:", len(train_ds), len(val_ds), len(test_ds))


In [ ]:
class BasicCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

def build_model(model_name, num_classes, pretrained=True):
    if model_name == "basic_cnn":
        return BasicCNN(num_classes)
    if model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model
    if model_name == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
        return model
    if model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        return model
    raise ValueError(model_name)


In [ ]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, y_true, y_pred, y_prob = 0.0, [], [], []

    for images, labels, _ in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        probs = torch.softmax(logits.detach(), dim=1)
        preds = probs.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

    y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "labels": y_true,
        "preds": y_pred,
        "probs": y_prob,
    }

def train_model(model_name, epochs=10, lr=1e-4):
    model = build_model(model_name, num_classes, pretrained=(model_name != "basic_cnn")).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    history, best_f1, best_state = [], -1, None

    for epoch in range(1, epochs + 1):
        print(f"\n[{model_name}] Epoch {epoch}/{epochs}")
        train_m = run_one_epoch(model, train_loader, criterion, optimizer)
        val_m = run_one_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step(val_m["f1"])
        row = {
            "model": model_name, "epoch": epoch,
            "train_loss": train_m["loss"], "train_accuracy": train_m["accuracy"], "train_f1": train_m["f1"],
            "val_loss": val_m["loss"], "val_accuracy": val_m["accuracy"], "val_f1": val_m["f1"],
        }
        history.append(row)
        print(row)
        if val_m["f1"] > best_f1:
            best_f1 = val_m["f1"]
            best_state = copy.deepcopy(model.state_dict())
            torch.save({"model_name": model_name, "state_dict": best_state,
                        "class_names": class_names, "image_size": IMAGE_SIZE},
                       MODEL_DIR / f"{model_name}_best.pt")
    model.load_state_dict(best_state)
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUTPUT_DIR / f"{model_name}_training_history.csv", index=False)
    return model, hist_df


In [ ]:
# For a quick test use EPOCHS = 2. For final run use 10-20.
MODELS_TO_TRAIN = ["basic_cnn", "resnet50"]
# Optional: MODELS_TO_TRAIN = ["basic_cnn", "resnet50", "densenet121", "efficientnet_b0"]
EPOCHS = 10

trained_models, histories = {}, {}
for name in MODELS_TO_TRAIN:
    trained_models[name], histories[name] = train_model(name, epochs=EPOCHS)


In [ ]:
for name, hist in histories.items():
    plt.figure(figsize=(8, 5))
    plt.plot(hist["epoch"], hist["train_loss"], label="Train loss")
    plt.plot(hist["epoch"], hist["val_loss"], label="Val loss")
    plt.title(f"{name} Loss Curve")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout()
    path = FIGURE_DIR / f"{name}_loss_curve.png"
    plt.savefig(path, dpi=200); plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(hist["epoch"], hist["train_accuracy"], label="Train accuracy")
    plt.plot(hist["epoch"], hist["val_accuracy"], label="Val accuracy")
    plt.title(f"{name} Accuracy Curve")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.tight_layout()
    path = FIGURE_DIR / f"{name}_accuracy_curve.png"
    plt.savefig(path, dpi=200); plt.show()


In [ ]:
def evaluate_model(model_name, model):
    criterion = nn.CrossEntropyLoss()
    m = run_one_epoch(model, test_loader, criterion, optimizer=None)
    y_true, y_pred, y_prob = m["labels"], m["preds"], m["probs"]

    try:
        y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
        auc = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")
    except Exception:
        auc = np.nan

    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    pd.DataFrame(report).transpose().to_csv(OUTPUT_DIR / f"{model_name}_classification_report.csv")

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(OUTPUT_DIR / f"{model_name}_confusion_matrix.csv")

    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{model_name} Confusion Matrix")
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right")
    plt.yticks(ticks, class_names)
    plt.xlabel("Predicted"); plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{model_name}_confusion_matrix.png", dpi=200)
    plt.show()

    pred_df = metadata[metadata["split"]=="test"].copy().reset_index(drop=True)
    pred_df["pred_label"] = y_pred
    pred_df["pred_class"] = [class_names[i] for i in y_pred]
    pred_df["confidence"] = y_prob.max(axis=1)
    for i, c in enumerate(class_names):
        pred_df[f"prob_{c}"] = y_prob[:, i]
    pred_df.to_csv(OUTPUT_DIR / f"{model_name}_test_predictions.csv", index=False)

    return {
        "model": model_name, "test_loss": m["loss"], "accuracy": m["accuracy"],
        "precision_weighted": m["precision"], "recall_weighted": m["recall"],
        "f1_weighted": m["f1"], "auc_macro_ovr": auc
    }

summaries = [evaluate_model(name, model) for name, model in trained_models.items()]
summary_df = pd.DataFrame(summaries)
summary_df.to_csv(OUTPUT_DIR / "classification_model_comparison.csv", index=False)
summary_df


## Outputs from Notebook 02

- `*_best.pt` model checkpoints
- training history CSV files
- training/validation curves
- classification report CSV files
- confusion matrix CSV and PNG files
- `classification_model_comparison.csv`

Next: run `03_xai_gradcam.ipynb`.
